# Train SentinelGrid's detector on WILDTRACK

This notebook fine-tunes a person detector using WILDTRACK's calibrated, synchronized camera frames and official multi-view annotations. In Colab choose **Runtime → Change runtime type → T4 GPU** before running.

Download the dataset from [Kaggle](https://www.kaggle.com/datasets/aryashah2k/large-scale-multicamera-detection-dataset) or the [official project](https://cvlab.epfl.ch/data/wildtrack/), unzip it into `/content/Wildtrack`, and ensure it has `Image_subsets/` and `annotations_positions/`. Do not use test frames while training.

In [1]:
!pip -q install kaggle

from google.colab import files
files.upload()  # Select the small kaggle.json file

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!mkdir -p /content/Wildtrack
!kaggle datasets download -d aryashah2k/large-scale-multicamera-detection-dataset -p /content/Wildtrack
!unzip -q /content/Wildtrack/*.zip -d /content/Wildtrack

!find /content/Wildtrack -maxdepth 2 -type d | head -20

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/aryashah2k/large-scale-multicamera-detection-dataset
License(s): CC0-1.0
100% 10.0G/10.0G [09:42<00:00, 18.5MB/s]

/content/Wildtrack
/content/Wildtrack/Wildtrack
/content/Wildtrack/Wildtrack/calibrations
/content/Wildtrack/Wildtrack/Image_subsets
/content/Wildtrack/Wildtrack/annotations_positions


In [2]:
!nvidia-smi

Sat Sep 12 11:54:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip -q install ultralytics==8.3.74 pyyaml

from pathlib import Path
import json, random, shutil
from collections import defaultdict
from PIL import Image
from ultralytics import YOLO

ROOT = Path("/content/Wildtrack/Wildtrack")
assert (ROOT / "Image_subsets").exists(), "Dataset folder not found"
assert (ROOT / "annotations_positions").exists(), "Annotations folder not found"

OUT = Path("/content/wildtrack_yolo")
shutil.rmtree(OUT, ignore_errors=True)

for split in ("train", "val"):
    (OUT / "images" / split).mkdir(parents=True)
    (OUT / "labels" / split).mkdir(parents=True)

print("Setup complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 623.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.7/914.7 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 92.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup complete.


In [4]:
ann_files = sorted((ROOT / "annotations_positions").glob("*.json"))
random.seed(42)
random.shuffle(ann_files)

cut = int(len(ann_files) * 0.8)
train_ids = {p.stem for p in ann_files[:cut]}
written = defaultdict(int)

for ann_path in ann_files:
    annotations = json.loads(ann_path.read_text())
    split = "train" if ann_path.stem in train_ids else "val"

    for view_idx in range(7):
        camera = f"C{view_idx + 1}"
        candidates = list((ROOT / "Image_subsets" / camera).glob(ann_path.stem + ".*"))

        if not candidates:
            continue

        image_path = candidates[0]

        with Image.open(image_path) as im:
            width, height = im.size

        lines = []

        for person in annotations:
            views = person.get("views", [])

            if len(views) <= view_idx:
                continue

            box = views[view_idx]

            if not box or box.get("xmin", -1) < 0:
                continue

            if box.get("xmax", -1) <= box.get("xmin", -1):
                continue

            x1 = max(0, min(box["xmin"], width))
            y1 = max(0, min(box["ymin"], height))
            x2 = max(0, min(box["xmax"], width))
            y2 = max(0, min(box["ymax"], height))

            if x2 <= x1 or y2 <= y1:
                continue

            xc = (x1 + x2) / 2 / width
            yc = (y1 + y2) / 2 / height
            bw = (x2 - x1) / width
            bh = (y2 - y1) / height

            lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

        name = f"{camera}_{ann_path.stem}"
        shutil.copy2(image_path, OUT / "images" / split / f"{name}{image_path.suffix}")
        (OUT / "labels" / split / f"{name}.txt").write_text("\n".join(lines))
        written[split] += 1

print("Images written:", dict(written))

Images written: {'train': 2240, 'val': 560}


In [5]:
yaml_path = OUT / "wildtrack.yaml"

yaml_path.write_text(f"""
path: {OUT}
train: images/train
val: images/val

names:
  0: person
""".strip())

print(yaml_path.read_text())

path: /content/wildtrack_yolo
train: images/train
val: images/val

names:
  0: person


In [6]:
detector = YOLO("yolo11n.pt")

results = detector.train(
    data=str(yaml_path),
    epochs=10,
    imgsz=640,
    batch=16,
    patience=3,
    device=0,
    workers=2,
    project="/content/runs",
    name="wildtrack_person",
    exist_ok=True,
    plots=True
)

100%|██████████| 5.35M/5.35M [00:00<00:00, 326MB/s]


New https://pypi.org/project/ultralytics/8.4.149 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.74 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/content/wildtrack_yolo/wildtrack.yaml, epochs=10, time=None, patience=3, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=/content/runs, name=wildtrack_person, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=No

100%|██████████| 755k/755k [00:00<00:00, 152MB/s]


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

train: Scanning /content/wildtrack_yolo/labels/train... 2240 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2240/2240 [00:25<00:00, 86.20it/s]


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      10/10       2.5G      1.095     0.9239     0.9686        198        640: 100%|██████████| 140/140 [02:00<00:00,  1.16it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 18/18 [00:24<00:00,  1.37s/it]


                   all        560       8360      0.869      0.756      0.873      0.552

10 epochs completed in 0.424 hours.
Optimizer stripped from /content/runs/wildtrack_person/weights/last.pt, 5.4MB
Optimizer stripped from /content/runs/wildtrack_person/weights/best.pt, 5.4MB

Validating /content/runs/wildtrack_person/weights/best.pt...
Ultralytics 8.3.74 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 238 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 18/18 [00:25<00:00,  1.41s/it]


                   all        560       8360       0.87      0.755      0.873      0.552
Speed: 0.2ms preprocess, 2.7ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to /content/runs/wildtrack_person


In [7]:
from google.colab import files

files.download("/content/runs/wildtrack_person/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
!find /content/Wildtrack/Wildtrack/calibrations -type f

/content/Wildtrack/Wildtrack/calibrations/intrinsic_zero/intr_CVLab4.xml
/content/Wildtrack/Wildtrack/calibrations/intrinsic_zero/intr_CVLab2.xml
/content/Wildtrack/Wildtrack/calibrations/intrinsic_zero/intr_IDIAP1.xml
/content/Wildtrack/Wildtrack/calibrations/intrinsic_zero/intr_CVLab1.xml
/content/Wildtrack/Wildtrack/calibrations/intrinsic_zero/intr_CVLab3.xml
/content/Wildtrack/Wildtrack/calibrations/intrinsic_zero/intr_IDIAP2.xml
/content/Wildtrack/Wildtrack/calibrations/intrinsic_zero/intr_IDIAP3.xml
/content/Wildtrack/Wildtrack/calibrations/extrinsic/extr_CVLab3.xml
/content/Wildtrack/Wildtrack/calibrations/extrinsic/extr_CVLab4.xml
/content/Wildtrack/Wildtrack/calibrations/extrinsic/extr_IDIAP1.xml
/content/Wildtrack/Wildtrack/calibrations/extrinsic/extr_CVLab2.xml
/content/Wildtrack/Wildtrack/calibrations/extrinsic/extr_IDIAP2.xml
/content/Wildtrack/Wildtrack/calibrations/extrinsic/extr_CVLab1.xml
/content/Wildtrack/Wildtrack/calibrations/extrinsic/extr_IDIAP3.xml
/content/Wild

In [9]:
print(open("/content/Wildtrack/Wildtrack/calibrations/intrinsic_original/intr_CVLab1.xml").read()[:1200])

print("\n--- EXTRINSIC ---\n")

print(open("/content/Wildtrack/Wildtrack/calibrations/extrinsic/extr_CVLab1.xml").read()[:1200])

<?xml version="1.0"?>
<opencv_storage>
<camera_matrix type_id="opencv-matrix">
  <rows>3</rows>
  <cols>3</cols>
  <dt>d</dt>
  <data>1743.4478759765625 0.0 934.5202026367188 0.0 1735.1566162109375 444.3987731933594 0.0 0.0 1.0</data></camera_matrix>
<distortion_coefficients type_id="opencv-matrix">
  <rows>5</rows>
  <cols>1</cols>
  <dt>d</dt>
  <data>-0.43248599767684937 0.6106230020523071 0.008233999833464622 0.0018599999602884054 -0.6923710107803345</data></distortion_coefficients>
</opencv_storage>


--- EXTRINSIC ---

<?xml version="1.0"?>
<opencv_storage>
  <rvec>
    1.759099006652832 0.46710100769996643 -0.331699013710022
   </rvec> 
  <tvec>
    -525.8941650390625 45.40763473510742 986.7235107421875
  </tvec>
</opencv_storage>



In [10]:
import cv2
import json
import shutil
import xml.etree.ElementTree as ET
import numpy as np
from pathlib import Path
from google.colab import files

camera_names = ["CVLab1", "CVLab2", "CVLab3", "CVLab4", "IDIAP1", "IDIAP2", "IDIAP3"]
package = Path("/content/T204_demo_package")

shutil.rmtree(package, ignore_errors=True)
(package / "config").mkdir(parents=True)
(package / "demo_frames").mkdir(parents=True)

def numbers(xml_file, xpath):
    root = ET.parse(xml_file).getroot()
    return np.fromstring(root.find(xpath).text, sep=" ")

calibrations = []

for index, camera_name in enumerate(camera_names, start=1):
    intrinsic = ROOT / "calibrations" / "intrinsic_original" / f"intr_{camera_name}.xml"
    extrinsic = ROOT / "calibrations" / "extrinsic" / f"extr_{camera_name}.xml"

    K = numbers(intrinsic, "camera_matrix/data").reshape(3, 3)
    rvec = numbers(extrinsic, "rvec").reshape(3, 1)
    tvec = numbers(extrinsic, "tvec").reshape(3, 1)

    R, _ = cv2.Rodrigues(rvec)
    ground_to_image = K @ np.column_stack((R[:, 0], R[:, 1], tvec.reshape(3)))

    # WILDTRACK uses centimetres; convert projected ground coordinates to metres.
    image_to_ground = np.diag([0.01, 0.01, 1.0]) @ np.linalg.inv(ground_to_image)

    calibrations.append({
        "camera_id": f"C{index}",
        "image_to_ground": image_to_ground.tolist()
    })

(package / "config" / "calibrations.json").write_text(
    json.dumps(calibrations, indent=2)
)

# Select one timestamp reserved for validation.
test_timestamp = next(p.stem for p in ann_files if p.stem not in train_ids)

for index in range(1, 8):
    source = next((ROOT / "Image_subsets" / f"C{index}").glob(test_timestamp + ".*"))
    shutil.copy2(source, package / "demo_frames" / f"C{index}_{test_timestamp}{source.suffix}")

zip_path = shutil.make_archive("/content/T204_demo_package", "zip", package)
print("Created:", zip_path)
print("Validation timestamp:", test_timestamp)
print("Files:", [p.name for p in (package / "demo_frames").iterdir()])

files.download(zip_path)

Created: /content/T204_demo_package.zip
Validation timestamp: 00000535
Files: ['C4_00000535.png', 'C7_00000535.png', 'C6_00000535.png', 'C3_00000535.png', 'C5_00000535.png', 'C2_00000535.png', 'C1_00000535.png']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Final evaluation for your report

Report the notebook's held-out 2D precision, recall, and mAP50. Then run the local fusion API on several validation timestamps using **all seven cameras**, compare its unified count with the number of records in that timestamp's official annotation JSON, and calculate MAE/RMSE. This is the proper evidence for the project objective; do not present detector mAP as multi-view counting accuracy.